In [1]:
import os
from PIL import Image

def convert_box_to_yolo(box_line, img_width, img_height):
    parts = box_line.strip().split(',', 8)  # 最多切 9 份,最后一份是文字(可能含逗号)
    coords = list(map(float, parts[:8]))

    xs = coords[0::2]  # 所有 x: x1,x2,x3,x4
    ys = coords[1::2]  # 所有 y: y1,y2,y3,y4

    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    x_center = (x_min + x_max) / 2 / img_width
    y_center = (y_min + y_max) / 2 / img_height
    width = (x_max - x_min) / img_width
    height = (y_max - y_min) / img_height

    return f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"


def convert_folder(img_dir, box_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    box_files = [f for f in os.listdir(box_dir) if f.endswith('.txt')]

    for box_file in box_files:
        name = os.path.splitext(box_file)[0]
        img_path = os.path.join(img_dir, name + '.jpg')

        if not os.path.exists(img_path):
            print(f"Skipping {name}: image not found")
            continue

        with Image.open(img_path) as img:
            img_width, img_height = img.size

        yolo_lines = []
        box_path = os.path.join(box_dir, box_file)
        try:
            with open(box_path, 'r', encoding='utf-8-sig') as f:
                lines = f.readlines()
        except UnicodeDecodeError:
            # 有些 SROIE 档案不是 UTF-8 编码,用比较宽松的编码 fallback 读取
            with open(box_path, 'r', encoding='cp1252', errors='ignore') as f:
                lines = f.readlines()

        for line in lines:
            if line.strip():
                yolo_lines.append(convert_box_to_yolo(line, box_file and img_width, img_height))

        with open(os.path.join(output_dir, name + '.txt'), 'w') as f:
            f.write('\n'.join(yolo_lines))

    print(f"Converted {len(box_files)} files → {output_dir}")


# 跑 train 资料夹
convert_folder(
    img_dir=r"C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data 训练\SROIE2019\train\img",
    box_dir=r"C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data 训练\SROIE2019\train\box",
    output_dir=r"C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data 训练\SROIE2019\train\labels"
)

# 跑 test 资料夹
convert_folder(
    img_dir=r"C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data 训练\SROIE2019\test\img",
    box_dir=r"C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data 训练\SROIE2019\test\box",
    output_dir=r"C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data 训练\SROIE2019\test\labels"
)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\Cht632\\Asgmnt RSW\\RSWY2S1\\AI\\ReceiptSplitter\\data 训练\\SROIE2019\\train\\box'

In [3]:
!pip install ultralytics easyocr streamlit pandas

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 10.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   -------------------------------- ------- 2.4/2.9 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 9.8 MB/s  0:00:00
   ---------------------------------------- 0.0/847.1 kB ? eta -:--:--
   ---------------------------------------- 847.1/847.1 kB 7.5 MB/s  0:00:00
   ---------------------------------------- 0.0/52.6 MB ? eta -:--:--
   - -------------------------------------- 2.4/52.6 MB 12.2 MB/s eta 0:00:05
   --- ------------------------------------ 5.0/52.6 MB 12.1 MB/s eta 0:00:04
   ----- ---------------------------------- 7.6/52.6 MB 12.1 MB/s eta 0:00:04
   ------- -------------------------------- 10.0/52.6 MB 11.9 MB/s eta 0:00:04
   --------- ------------------------------ 12.6/52.6 MB 12.0 MB/s eta 0:00:04
   ---------- --------------

In [10]:
import yaml

data = {
    'train': r'C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data\SROIE2019\train\img',
    'val': r'C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data\SROIE2019\test\img',
    'nc': 1,
    'names': ['text']
}

yaml_path = r'C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\dataset.yaml'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("dataset.yaml created!")
print(open(yaml_path).read())

dataset.yaml created!
names:
- text
nc: 1
train: C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data\SROIE2019\train\img
val: C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\data\SROIE2019\test\img



In [19]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data=r'C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=8,
    name='receipt_yolo',
    dropout=0.2,
    weight_decay=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,
    scale=0.5,
    translate=0.1,
    patience=20,
    save_period=10,
)

print("Training done!")

Ultralytics 8.4.115  Python-3.11.15 torch-2.13.0+cpu CPU (13th Gen Intel Core i5-13420H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Cht632\Asgmnt RSW\RSWY2S1\AI\ReceiptSplitter\dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=